## Integer Programming: Determine Facility Locations

A company wants to build distribution centers at some candidate locations to serve given markets. Each distribution center (if opened) has an operating cost and shipping capacity. There are also shipping costs from each facility to each market. Our goal is to choose some subset of locations to build distribution centers, to meet the demand of each market, and minimize the total cost.

This can be formulated as:

$$
\begin{array}{rlll}
    \min & \displaystyle \sum_{c \in C} o_cf_c + \sum_{c \in C,m \in M} s_{cm}e_{cm} & & \text{minimize operating costs + shipping costs} \\
    \text{s.t.}
        & \displaystyle \sum_{c \in C} e_{cm} \ge d_m & \forall m \in M & \text{incoming deliveries >= market demand} \\[15pt]
        & \displaystyle \sum_{m \in M} e_{cm} \le t_cf_c & \forall c \in C & \text{outcoming deliveries <= total city capacity} \\[15pt]
        & f_{c} \in \{0, 1\} &\forall c \in C & \text{0 or 1 facilities per location} \\[5pt]
        & e_{cm} \ge 0 &\forall c \in C,m \in M & \text{delivery can be any positive amount, or zero}
\end{array}
$$

with variables defined as follows:
- $f_c$: whether city $c$ has a facility
- $e_{cm}$: how much is delivered from facility in city $c$ to market $m$
- $s_{cm}$: the per-unit shipping costs from facility in city $c$ to market $m$
- $d_m$: the demand of market $m$
- $o_c$: the constant operating costs of facility in city $c$
- $t_c$: the total shipping capacity of facility in city $c$
- $C$: the set of cities
- $M$: the set of markets

Problem described in 3.13 of [Operations Research (1): Models and Applications](https://www.coursera.org/learn/operations-research-modeling/home/welcome).


In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import LinearConstraint, milp

city_info = pd.DataFrame([
    ("Spokane, WA", 40000, 20000),
    ("Reno, NV", 30000, 20000),
    ("Omahe, NE", 25000, 15000),
    ("Harrisburg, PA", 40000, 25000),
    ("Jacksonville, FL", 30000, 15000),
], columns=("city", "operating_cost", "capacity"))

market_info = pd.DataFrame([
    ("Northwest", 8000),
    ("Southwest", 12000),
    ("Midwest", 9000),
    ("Southeast", 14000),
    ("Northeast", 17000),
], columns=("market", "demand"))

shipping_cost = pd.DataFrame([
    ("Northwest", 2.4, 3.25, 4.05, 5.25, 6.95),
    ("Southwest", 3.5, 2.3, 3.25, 6.05, 5.85),
    ("Midwest", 4.8, 3.4, 2.85, 4.3, 4.8),
    ("Southeast", 6.8, 5.25, 4.3, 3.25, 2.1),
    ("Northeast", 5.75, 6, 4.75, 2.75, 3.5),
], columns=["market"] + list(np.arange(5)))

display(city_info)
display(market_info)
print("Shipping costs:")
df = shipping_cost.copy()
df.columns = ["market"] + list(city_info.city)
display(df)

,city,operating_cost,capacity
0,"Spokane, WA",40000,20000
1,"Reno, NV",30000,20000
2,"Omahe, NE",25000,15000
3,"Harrisburg, PA",40000,25000
4,"Jacksonville, FL",30000,15000


,market,demand
0,Northwest,8000
1,Southwest,12000
2,Midwest,9000
3,Southeast,14000
4,Northeast,17000


Shipping costs:


,market,"Spokane, WA","Reno, NV","Omahe, NE","Harrisburg, PA","Jacksonville, FL"
0,Northwest,2.40,3.25,4.05,5.25,6.95
1,Southwest,3.50,2.30,3.25,6.05,5.85
2,Midwest,4.80,3.40,2.85,4.30,4.80
3,Southeast,6.80,5.25,4.30,3.25,2.10
4,Northeast,5.75,6.00,4.75,2.75,3.50


In [2]:
# Set objective and constraints

# Variables:
# [
#    facility in city 1 to n,
#    how much to deliver from city i to district j,
# ]

cities = len(city_info)
markets = len(market_info)

# Minimise sum of operating costs and shipping costs
coef = np.concat([city_info.operating_cost, shipping_cost.iloc[:, shipping_cost.columns != "market"].values.flatten()])

# Convenience function to get index of variable for delivery from a given city to a given market
def delivery(from_city, to_market):
    return cities + cities * to_market + from_city

var_count = len(coef)

constraints = []

# The sum of incoming deliveries/edges for each market must meet their demand
for to_market in range(markets):
    A_curr = np.zeros(var_count)
    for from_city in range(cities):
        A_curr[delivery(from_city, to_market)] = 1
    demand = market_info.demand[to_market]
    constraints.append(LinearConstraint(A_curr, lb=demand))

# The sum of outcoming deliveries/edges for each city must not exceeed their capacity
# facility * capacity >= sum of outgoing deliveries, or facility * capacity - sum of outgoing deliveries >= 0
for from_city in range(cities):
    A_curr = np.zeros(var_count)
    A_curr[from_city] = city_info.capacity[from_city]
    for to_market in range(markets):
        A_curr[delivery(from_city, to_market)] = -1
    constraints.append(LinearConstraint(A_curr, lb=0))

# Make sure we only assign 0 or 1 facility per city
for city in range(cities):
    A_curr = np.zeros(var_count)
    A_curr[city] = 1
    constraints.append(LinearConstraint(A_curr, 0, 1))

In [3]:
# Run solver

# Set all variables to be integers
integrality = np.ones_like(coef)

res = milp(c=coef, integrality=integrality, constraints=constraints)
res

        message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
        success: True
         status: 0
            fun: 268950.0
              x: [-0.000e+00  1.000e+00 ...  1.700e+04  0.000e+00]
 mip_node_count: 1
 mip_dual_bound: 268950.0
        mip_gap: 0.0

In [4]:
print(f"For an optimal cost of {res.fun:g}:")
print("Build facilities in the following cities:")
display(city_info[["city"]][res.x[:cities] > 0])

print("Deliver the following quantities from the given city to the given market:")
pd.DataFrame(res.x[cities:].reshape(markets, cities).round(), columns=city_info.city, index=market_info.market).map(int)

For an optimal cost of 268950:
Build facilities in the following cities:


,city
1,"Reno, NV"
3,"Harrisburg, PA"
4,"Jacksonville, FL"


Deliver the following quantities from the given city to the given market:


city,"Spokane, WA","Reno, NV","Omahe, NE","Harrisburg, PA","Jacksonville, FL"
market,,,,,
Northwest,0,8000,0,0,0
Southwest,0,12000,0,0,0
Midwest,0,0,0,8000,1000
Southeast,0,0,0,0,14000
Northeast,0,0,0,17000,0
